# ABot-Recon box — call it & see the results

Direct gRPC call to the `abot_recon` box (streaming 3D reconstruction) with `visionist_client`.
Two calls, mirroring the box's own design:

- **`trajectory`** — camera poses only (cheap, no point maps) → the **camera path**
- **`reconstruct`** — poses + per-frame point maps → the **3D point cloud**

Point maps are heavy (~1.7 MB/frame each for `local_points` and `world_points` on the fixed
280×504 grid), so keep the `reconstruct` frame count modest; use `trajectory` for the cheap,
full-length pose pass.

Visualisations are **interactive** (Plotly): drag to orbit, scroll to zoom, hover for frame index.


In [13]:
from visionist_client import Visionist, load
import numpy as np
import plotly.graph_objects as go
import cv2

BOX   = "localhost:9078"                # abot_recon  (registry/ports.json)
VIDEO = "/home/manuelf/room.mp4"

TRAJ_FRAMES = 128     # poses-only call - cheap; raise to 256 for the full path
REC_FRAMES  = 16      # reconstruct - point maps are ~1.7 MB/frame, keep modest

H, W = 280, 504       # the model's fixed output grid (world_points is [F, H, W, 3])
ASPECT = W / H           # frustum image aspect (504x280)
FRUSTUM_SCALE = 0.05    # camera-frustum size = scene_span * this   (lower = smaller pyramids)

b = Visionist(BOX, config_key="abot_recon")
print("box:", b.info())


def _maybe_show(fig):
    """Render inline inside a notebook; return the figure otherwise."""
    try:
        from IPython import get_ipython
        if get_ipython() is not None:
            fig.show()
    except Exception:
        pass
    return fig


def _fig3d(title):
    fig = go.Figure()
    fig.update_layout(
        title=title, template="plotly_white",
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
        margin=dict(l=0, r=0, t=46, b=0), width=780, height=640,
    )
    return fig


def _cam_frame(i, P, k):
    """World-space apex + 4 image-plane corners of camera ``i`` (OpenCV: x right, y down, z forward)."""
    R, t = P[i, :3, :3], P[i, :3, 3]
    z, x, y = R[:, 2], R[:, 0], R[:, 1]
    d = 2.0 * k
    hw, hh = k * ASPECT, k
    def C(sx, sy):
        return t + d * z + sx * hw * x + sy * hh * y
    return t, [C(-1, -1), C(1, -1), C(1, 1), C(-1, 1)]


def _frustum_traces(P, scale, color="rgba(255,255,255,0.85)", width=3):
    """All cameras' frustum edges (image-plane rectangle + apex projections), one broken-lines trace."""
    X, Y, Z = [], [], []
    for i in range(len(P)):
        t, cs = _cam_frame(i, P, scale)
        edges = [(cs[0], cs[1]), (cs[1], cs[2]), (cs[2], cs[3]), (cs[3], cs[0]),
                 (t, cs[0]), (t, cs[1]), (t, cs[2]), (t, cs[3])]
        for a, bb in edges:
            X += [a[0], bb[0], None]; Y += [a[1], bb[1], None]; Z += [a[2], bb[2], None]
    return go.Scatter3d(x=X, y=Y, z=Z, mode="lines", line=dict(width=width, color=color),
                        name="frustum", showlegend=True, hoverinfo="skip")


def sample_frame_colors(video, max_frames, frame_step=1, H=H, W=W):
    """Decode the SAME frames the box used (first ``max_frames``, every ``frame_step``-th) and apply
    the model's exact input preprocessing (width-lock resize to 504, then centre-crop/pad to 280).
    Returns (F, H, W, 3) uint8 RGB - rgb[f, h, w] is the true colour of world-point (f, h, w)."""
    cap = cv2.VideoCapture(str(video))
    frames, index = [], 0
    while len(frames) < max_frames:
        ok, bgr = cap.read()
        if not ok:
            break
        if index % max(1, frame_step) == 0:
            frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
        index += 1
    cap.release()
    if not frames:
        raise RuntimeError("video decoded to zero frames")
    out = []
    for rgb in frames:
        h0, w0 = rgb.shape[:2]
        rh = max(1, round(h0 * W / max(w0, 1)))
        img = cv2.resize(rgb, (W, rh), interpolation=cv2.INTER_CUBIC)   # (rh, W, 3)
        if img.shape[0] > H:
            top = round((img.shape[0] - H) * 0.5); img = img[top:top + H, :]
        elif img.shape[0] < H:
            top = (H - img.shape[0]) // 2
            canvas = np.full((H, W, 3), 128, np.uint8)
            canvas[top:top + img.shape[0], :] = img; img = canvas
        out.append(img)
    return np.stack(out, axis=0)


def plot_path(P, title="camera path", frustum=FRUSTUM_SCALE):
    """Interactive 3D camera path + frustums.  Colour = frame order (first -> last)."""
    P = np.asarray(P, float); F = len(P)
    c = P[:, :3, 3]
    span = max(float((c.max(0) - c.min(0)).max()), 1e-6)
    fig = _fig3d(title)
    fig.add_trace(go.Scatter3d(x=c[:,0], y=c[:,1], z=c[:,2], mode="lines",
                               line=dict(width=4, color="rgba(120,120,120,0.85)"), name="path", showlegend=True))
    fig.add_trace(go.Scatter3d(x=c[:,0], y=c[:,1], z=c[:,2], mode="markers",
                               marker=dict(size=6, color=np.arange(F), colorscale="Viridis",
                                           colorbar=dict(title="frame", len=0.45)),
                               name="cameras", showlegend=False,
                               text=[i for i in range(F)], hovertemplate="frame %{text}<extra></extra>"))
    fig.add_trace(_frustum_traces(P, span * frustum))
    return _maybe_show(fig)


def plot_scene(P, world_points, rgb=None, max_pts=60000, seed=0, frustum=FRUSTUM_SCALE):
    """Interactive 3D point cloud + camera frustums.
    With ``rgb`` (F,H,W,3): points show their TRUE pixel colour.  Otherwise: colour = source frame."""
    P   = np.asarray(P, float); F = P.shape[0]
    wp  = np.asarray(world_points, float).reshape(F, -1, 3)
    nf  = wp.shape[1]
    flat = wp.reshape(-1, 3); frame_of = np.repeat(np.arange(F), nf)
    idx  = np.random.default_rng(seed).choice(flat.shape[0], size=min(max_pts, flat.shape[0]), replace=False)
    pts  = flat[idx]
    span = max(float((flat.max(0) - flat.min(0)).max()), 1e-6)
    if rgb is not None:
        rgbf = np.asarray(rgb, np.uint8).reshape(F, nf, 3).reshape(-1, 3)[idx]
        cols = ["#%02x%02x%02x" % (int(r), int(g), int(bl)) for r, g, bl in rgbf]
        marker = dict(size=2.4, color=cols, opacity=0.9)
        title = f"world point maps \u00b7 {F} frames \u00b7 colour = TRUE pixel RGB"
    else:
        t = frame_of[idx] / max(F - 1, 1)
        marker = dict(size=2.4, color=t, colorscale="Viridis", opacity=0.85, colorbar=dict(title="frame", len=0.5))
        title = f"world point maps \u00b7 {F} frames \u00b7 colour = source frame"
    fig = _fig3d(title)
    fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode="markers", marker=marker,
                               name="points", showlegend=True,
                               text=[int(f) for f in frame_of[idx]], hovertemplate="frame %{text}<extra></extra>"))
    c = P[:, :3, 3]
    fig.add_trace(go.Scatter3d(x=c[:,0], y=c[:,1], z=c[:,2], mode="lines",
                               line=dict(width=4, color="rgba(0,0,0,0.9)"), name="path", showlegend=False))
    fig.add_trace(_frustum_traces(P, span * frustum, color="rgba(0,0,0,0.8)"))
    return _maybe_show(fig)


box: {'service': 'pipeline.PipelineService', 'methods': ['Process'], 'reflection': True, 'reachable': True}


### Call contract (what the box actually reads)

```python
# NOTE: parameters are NESTED under "parameters" (a flat key is silently ignored):
config = {"abot_recon": {
    "command": "reconstruct",            # or "trajectory" | "reset"
    "parameters": {
        "max_frames": int,               # >= 3;  stream cap 22000
        "frame_step": int,               # keep every Nth frame while decoding
        "dense_stride": int,             # reconstruct: point maps every Nth frame
        "confidence_threshold": float,   # 0 = keep all
        "device": "cuda",                # or "cpu" (impractical at ~1.0 B params)
    },
}}
data   = {"video": <bytes | Path>}       # OR {"images": [bytes, ...]}  - not both
```


## 1. Camera path (trajectory)

In [14]:
# ---- 1. TRAJECTORY (poses only - cheap, no point maps) --------------------
cfg   = {"abot_recon": {"command": "trajectory",
                        "parameters": {"max_frames": TRAJ_FRAMES, "frame_step": 1}}}
res_t = b.run(data={"video": load(VIDEO)}, config=cfg)
sec   = res_t.config["abot_recon"]
P     = np.asarray(res_t.camera_poses, float)
print("status", sec.get("status"), "| frames", sec.get("num_frames"),
      "| path length", round(float(np.linalg.norm(np.diff(P[:, :3, 3], axis=0), axis=1).sum()), 4))
print("fields:", sorted(res_t.fields))
plot_path(P, f"camera path \u00b7 {P.shape[0]} frames  (trajectory)")


status done | frames 128 | path length 1.7061
fields: ['camera_poses', 'camera_poses_noloop', 'relative_poses', 'relative_poses_noloop']


## 2. Point cloud (reconstruct) — the heavy call

The cloud is coloured by each point's **true pixel RGB** (sampled from the same input frames the box decoded).


In [15]:
# ---- 2. RECONSTRUCT (poses + per-frame point maps) ------------------------
cfg = {"abot_recon": {"command": "reconstruct",
                      "parameters": {"max_frames": TRAJ_FRAMES,
                                     "frame_step": 10, "dense_stride": 1}}}
res = b.run(data={"video": load(VIDEO)}, config=cfg)
sec = res.config["abot_recon"]
print("status", sec.get("status"),
      "| frames", sec.get("num_frames"),
      "| dense frames", sec.get("num_dense_frames"))
print(f"\n{'field':18s} {'shape':22s} {'dtype':7s}  size")
for k in sorted(res.fields):
    v = res.fields[k]
    if isinstance(v, np.ndarray):
        print(f"{k:18s} {str(tuple(v.shape)):22s} {str(v.dtype):7s} {v.nbytes/1e6:8.1f} MB")
    else:
        print(f"{k:18s} {v!r}")


status done | frames 38 | dense frames 38

field              shape                  dtype    size
camera_poses       (38, 4, 4)             float32      0.0 MB
camera_poses_noloop (38, 4, 4)             float32      0.0 MB
confidence         (38, 280, 504)         float32     21.5 MB
confidence_mask    (38, 280, 504)         bool         5.4 MB
local_points       (38, 280, 504, 3)      float32     64.4 MB
relative_poses     (37, 4, 4)             float32      0.0 MB
relative_poses_noloop (37, 4, 4)             float32      0.0 MB
world_points       (38, 280, 504, 3)      float32     64.4 MB


In [17]:
# ---- 3. POINT CLOUD + CAMERAS ---------------------------------------------
# The box returns POSITIONS only; to show each point's TRUE pixel colour we sample
# the same input frames the box decoded and re-apply the 280x504 grid mapping,
# so rgb[f, h, w] is exactly the colour of world-point (f, h, w).
wp = res.fields.get("world_points")
P  = np.asarray(res.camera_poses, float)
if wp is not None:
    rgb = sample_frame_colors(VIDEO, wp.shape[0], frame_step=10)
    print("pixel colours:", tuple(rgb.shape), rgb.dtype, "for", int(rgb.shape[0]), "frames")
    plot_scene(P, wp, rgb=rgb, max_pts=60000, frustum=FRUSTUM_SCALE*0.1)   # true pixel RGB; smaller: frustum=0.03
    # plot_scene(P, wp, rgb=None, max_pts=60000)   # colour-by-frame fallback
else:
    print("world_points not returned (output_world_points is True by default)")


pixel colours: (38, 280, 504, 3) uint8 for 38 frames


In [5]:
# ---- 4. PEEL AT THE NUMBERS ------------------------------------------------
P   = np.asarray(res.camera_poses, float)
R01 = np.asarray(res.relative_poses, float)[0]
print("frame-0 pose (expected identity - world is anchored to frame 0):")
print(P[0].round(4))
print("\nrelative pose frame0 -> frame1 (one fewer row than camera_poses):")
print(R01.round(4))

conf = np.asarray(res.confidence)
print(f"\nconfidence: min {conf.min():.3f}  mean {conf.mean():.3f}  max {conf.max():.3f}")
lm   = np.asarray(res.confidence_mask)
print("kept fraction (confidence_mask):", round(float(lm.mean()), 4))


frame-0 pose (expected identity - world is anchored to frame 0):
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

relative pose frame0 -> frame1 (one fewer row than camera_poses):
[[ 1.0e+00  1.0e-04 -0.0e+00  1.5e-03]
 [-1.0e-04  1.0e+00  1.5e-03  6.0e-04]
 [ 0.0e+00 -1.5e-03  1.0e+00 -2.0e-03]
 [ 0.0e+00  0.0e+00  0.0e+00  1.0e+00]]

confidence: min 0.004  mean 0.643  max 0.988
kept fraction (confidence_mask): 1.0


In [6]:
# ---- tidy up ----------------------------------------------------------------
print("reset:", b.reset().config)     # clear box-side state (best-effort)
b.close()
print("done")


reset: {'abot_recon': {'status': 'done', 'action': 'reset'}}
done
